In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import nibabel as nib

import tensorflow as tf

from scipy.ndimage import binary_fill_holes
from scipy.spatial.distance import cdist


print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s):", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(
                gpu,
                True
            )

        print("✓ GPU memory growth enabled")

    except RuntimeError as e:
        print("GPU memory-growth warning:", e)

TensorFlow: 2.20.0
GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU memory growth enabled


In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("✓ TensorFlow deterministic operations enabled")

except Exception as e:
    print("Determinism note:", e)

print("Seed:", SEED)

✓ TensorFlow deterministic operations enabled
Seed: 42


In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE1_DIR = (
    PROJECT_DIR /
    "phase1_segmentation"
)

P1A_DIR = (
    PHASE1_DIR /
    "phase1A_resolution"
)

P1B_DIR = (
    PHASE1_DIR /
    "phase1B_hyperparameters"
)

P1C_DIR = (
    PHASE1_DIR /
    "phase1C_capacity"
)

P1D_DIR = (
    PHASE1_DIR /
    "phase1D_loss"
)

P1D_CONFIG_DIR = (
    P1D_DIR /
    "configs"
)

P1D_MODEL_DIR = (
    P1D_DIR /
    "models"
)

P1D_HISTORY_DIR = (
    P1D_DIR /
    "histories"
)

P1D_TABLE_DIR = (
    P1D_DIR /
    "tables"
)

P1D_PLOT_DIR = (
    P1D_DIR /
    "plots"
)


for folder in [
    P1D_CONFIG_DIR,
    P1D_MODEL_DIR,
    P1D_HISTORY_DIR,
    P1D_TABLE_DIR,
    P1D_PLOT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("Phase 1D directory:")
print(P1D_DIR)

print("\n✓ Phase 1D folders ready.")

Phase 1D directory:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss

✓ Phase 1D folders ready.


In [ ]:
# ================================================================
# PHASE 1A — RESOLUTION
# ================================================================

resolution_path = (
    P1A_DIR /
    "configs" /
    "phase1A_final_resolution_decision.json"
)

assert resolution_path.exists()

with open(resolution_path, "r") as f:
    resolution_decision = json.load(f)

SELECTED_RESOLUTION = int(
    resolution_decision["selected_resolution"]
)


# ================================================================
# PHASE 1B — HYPERPARAMETERS
# ================================================================

P1B_CONFIG_DIR = (
    P1B_DIR /
    "configs"
)


with open(
    P1B_CONFIG_DIR /
    "phase1B1_learning_rate_decision.json",
    "r"
) as f:
    lr_decision = json.load(f)

SELECTED_LEARNING_RATE = float(
    lr_decision["selected_learning_rate"]
)


with open(
    P1B_CONFIG_DIR /
    "phase1B2_optimizer_decision.json",
    "r"
) as f:
    optimizer_decision = json.load(f)

SELECTED_OPTIMIZER = (
    optimizer_decision[
        "selected_optimizer"
    ]
)


with open(
    P1B_CONFIG_DIR /
    "phase1B3_batch_size_decision.json",
    "r"
) as f:
    batch_decision = json.load(f)

SELECTED_BATCH_SIZE = int(
    batch_decision[
        "selected_batch_size"
    ]
)


with open(
    P1B_CONFIG_DIR /
    "phase1B4_augmentation_decision.json",
    "r"
) as f:
    augmentation_decision = json.load(f)

SELECTED_AUGMENTATION = (
    augmentation_decision[
        "selected_augmentation"
    ]
)


# ================================================================
# PHASE 1C — CAPACITY
# ================================================================

P1C_CONFIG_DIR = (
    P1C_DIR /
    "configs"
)

capacity_path = (
    P1C_CONFIG_DIR /
    "phase1C_capacity_decision.json"
)

assert capacity_path.exists()

with open(capacity_path, "r") as f:
    capacity_decision = json.load(f)


SELECTED_CAPACITY = (
    capacity_decision[
        "selected_capacity"
    ]
)

SELECTED_FILTERS = (
    capacity_decision[
        "selected_filters"
    ]
)

SELECTED_PARAMETER_COUNT = int(
    capacity_decision[
        "selected_parameter_count"
    ]
)


# ================================================================
# VERIFY LOCKED PIPELINE
# ================================================================

assert SELECTED_RESOLUTION == 256
assert np.isclose(
    SELECTED_LEARNING_RATE,
    1e-4
)
assert SELECTED_OPTIMIZER == "Adam"
assert SELECTED_BATCH_SIZE == 2
assert SELECTED_AUGMENTATION is False
assert SELECTED_CAPACITY == "large"
assert SELECTED_PARAMETER_COUNT == 7_759_521


print("=" * 70)
print("LOCKED PIPELINE ENTERING PHASE 1D")
print("=" * 70)

print(
    f"Resolution    : "
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    f"Learning rate : "
    f"{SELECTED_LEARNING_RATE}"
)

print(
    f"Optimizer     : "
    f"{SELECTED_OPTIMIZER}"
)

print(
    f"Batch size    : "
    f"{SELECTED_BATCH_SIZE}"
)

print("Augmentation  : None")

print(
    f"Capacity      : "
    f"{SELECTED_CAPACITY}"
)

print(
    f"Filters       : "
    f"{SELECTED_FILTERS}"
)

print(
    f"Parameters    : "
    f"{SELECTED_PARAMETER_COUNT:,}"
)

print("\nACTIVE VARIABLE: LOSS FUNCTION")

print("\n✓ Previous decisions verified.")

LOCKED PIPELINE ENTERING PHASE 1D
Resolution    : 256×256
Learning rate : 0.0001
Optimizer     : Adam
Batch size    : 2
Augmentation  : None
Capacity      : large
Filters       : [32, 64, 128, 256, 512]
Parameters    : 7,759,521

ACTIVE VARIABLE: LOSS FUNCTION

✓ Previous decisions verified.


In [ ]:
# CELL 6A — ONLY IF DATASET IS MISSING

import zipfile

OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR /
    "download"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)

DATASET_PATH = (
    INNER_EXTRACT_DIR /
    "database_nifti"
)


if not DATASET_PATH.exists():

    assert OUTER_ZIP_PATH.exists(), (
        f"Archive not found: {OUTER_ZIP_PATH}"
    )

    if not INNER_ARCHIVE_PATH.exists():

        print("Extracting outer archive...")

        OUTER_EXTRACT_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            OUTER_ZIP_PATH,
            "r"
        ) as archive:

            archive.extractall(
                OUTER_EXTRACT_DIR
            )

        print("✓ Outer archive extracted.")


    print("Extracting inner CAMUS archive...")

    INNER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        INNER_ARCHIVE_PATH,
        "r"
    ) as archive:

        archive.extractall(
            INNER_EXTRACT_DIR
        )


assert DATASET_PATH.exists()

print("\n✓ CAMUS dataset ready:")
print(DATASET_PATH)

Extracting outer archive...
✓ Outer archive extracted.
Extracting inner CAMUS archive...

✓ CAMUS dataset ready:
/content/camus_data_final/database_nifti


In [ ]:
SPLIT_PATH = (
    PROJECT_DIR /
    "splits" /
    "patient_split_seed42.json"
)

assert SPLIT_PATH.exists()


with open(SPLIT_PATH, "r") as f:
    patient_split = json.load(f)


train_ids = patient_split.get(
    "train_ids",
    patient_split.get("train")
)

val_ids = patient_split.get(
    "val_ids",
    patient_split.get(
        "validation",
        patient_split.get("val")
    )
)

test_ids = patient_split.get(
    "test_ids",
    patient_split.get("test")
)


assert len(train_ids) == 400
assert len(val_ids) == 50
assert len(test_ids) == 50

assert not (
    set(train_ids) & set(val_ids)
)

assert not (
    set(train_ids) & set(test_ids)
)

assert not (
    set(val_ids) & set(test_ids)
)


TRAIN_FRAMES = len(train_ids) * 2
VAL_FRAMES = len(val_ids) * 2


print("Train patients     :", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients      :", len(test_ids))

print("Training frames    :", TRAIN_FRAMES)
print("Validation frames  :", VAL_FRAMES)

print("\n✓ Patient split verified.")
print("✓ Test set excluded.")

Train patients     : 400
Validation patients: 50
Test patients      : 50
Training frames    : 800
Validation frames  : 100

✓ Patient split verified.
✓ Test set excluded.


In [ ]:
MAX_EPOCHS = 30
THRESHOLD = 0.50


LOSS_CONFIGS = {

    "bce_dice": {
        "description":
            "Binary cross-entropy + Dice loss"
    },

    "focal_dice": {
        "description":
            "Focal loss + Dice loss",

        "focal_alpha":
            0.25,

        "focal_gamma":
            2.0
    },

    "tversky": {
        "description":
            "Standalone Tversky loss",

        "tversky_alpha":
            0.5,

        "tversky_beta":
            0.5
    }
}


print("=" * 70)
print("PHASE 1D — CONTROLLED LOSS FUNCTION EXPERIMENT")
print("=" * 70)

print("\nLOCKED")

print(
    "Resolution        :",
    f"{SELECTED_RESOLUTION}×{SELECTED_RESOLUTION}"
)

print(
    "Architecture      : Large U-Net"
)

print(
    "Parameters        :",
    f"{SELECTED_PARAMETER_COUNT:,}"
)

print(
    "Learning rate     :",
    SELECTED_LEARNING_RATE
)

print(
    "Optimizer         :",
    SELECTED_OPTIMIZER
)

print(
    "Batch size        :",
    SELECTED_BATCH_SIZE
)

print(
    "Augmentation      : None"
)

print(
    "Epoch budget      :",
    MAX_EPOCHS
)

print(
    "Threshold         :",
    THRESHOLD
)


print("\nONLY VARIABLE")
print("Loss function")


print("\nLOSS ARMS")

for name, config in LOSS_CONFIGS.items():

    print(
        f"{name:>12} : "
        f"{config['description']}"
    )


print("\nFIXED LOSS PARAMETERS")
print("Focal alpha : 0.25")
print("Focal gamma : 2.0")
print("Tversky α   : 0.5")
print("Tversky β   : 0.5")


print("\nDATA")
print(
    "Training frames   :",
    TRAIN_FRAMES
)
print(
    "Validation frames :",
    VAL_FRAMES
)
print(
    "Test used         : NO"
)

print("\n✓ Phase 1D design defined.")

PHASE 1D — CONTROLLED LOSS FUNCTION EXPERIMENT

LOCKED
Resolution        : 256×256
Architecture      : Large U-Net
Parameters        : 7,759,521
Learning rate     : 0.0001
Optimizer         : Adam
Batch size        : 2
Augmentation      : None
Epoch budget      : 30
Threshold         : 0.5

ONLY VARIABLE
Loss function

LOSS ARMS
    bce_dice : Binary cross-entropy + Dice loss
  focal_dice : Focal loss + Dice loss
     tversky : Standalone Tversky loss

FIXED LOSS PARAMETERS
Focal alpha : 0.25
Focal gamma : 2.0
Tversky α   : 0.5
Tversky β   : 0.5

DATA
Training frames   : 800
Validation frames : 100
Test used         : NO

✓ Phase 1D design defined.


In [ ]:
def load_nifti_2d(file_path):

    nii = nib.load(
        str(file_path)
    )

    data = np.asarray(
        nii.get_fdata(),
        dtype=np.float32
    )

    if (
        data.ndim == 3
        and data.shape[-1] == 1
    ):
        data = data[:, :, 0]

    if data.ndim != 2:
        raise ValueError(
            f"Expected 2D image, got {data.shape}"
        )

    return data


def load_camus_frame(
    patient_id,
    phase
):

    patient_dir = (
        DATASET_PATH /
        patient_id
    )

    image = load_nifti_2d(
        patient_dir /
        f"{patient_id}_4CH_{phase}.nii.gz"
    )

    full_mask = load_nifti_2d(
        patient_dir /
        f"{patient_id}_4CH_{phase}_gt.nii.gz"
    )

    lv_mask = (
        full_mask == 1
    ).astype(np.float32)

    return image, lv_mask


def preprocess_frame(
    image,
    mask,
    resolution=SELECTED_RESOLUTION
):

    image_resized = cv2.resize(
        image,
        (
            resolution,
            resolution
        ),
        interpolation=cv2.INTER_LINEAR
    )

    mask_resized = cv2.resize(
        mask,
        (
            resolution,
            resolution
        ),
        interpolation=cv2.INTER_NEAREST
    )


    minimum = float(
        image_resized.min()
    )

    maximum = float(
        image_resized.max()
    )


    if maximum > minimum:

        image_resized = (
            image_resized - minimum
        ) / (
            maximum - minimum
        )

    else:

        image_resized = np.zeros_like(
            image_resized,
            dtype=np.float32
        )


    image_resized = (
        image_resized.astype(
            np.float32
        )
    )

    mask_resized = (
        mask_resized > 0.5
    ).astype(np.float32)


    return (
        image_resized[..., None],
        mask_resized[..., None]
    )


print("✓ Loader and preprocessing ready.")

✓ Loader and preprocessing ready.


In [ ]:
def frame_generator(
    patient_ids,
    shuffle=True,
    seed=SEED
):

    frame_list = [
        (pid, phase)
        for pid in patient_ids
        for phase in ["ED", "ES"]
    ]

    rng = np.random.default_rng(
        seed
    )


    while True:

        indices = np.arange(
            len(frame_list)
        )

        if shuffle:
            rng.shuffle(indices)


        for index in indices:

            pid, phase = (
                frame_list[index]
            )

            image, mask = (
                load_camus_frame(
                    pid,
                    phase
                )
            )

            x, y = preprocess_frame(
                image,
                mask
            )

            yield (
                x.astype(np.float32),
                y.astype(np.float32)
            )


OUTPUT_SIGNATURE = (

    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    ),

    tf.TensorSpec(
        shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        ),
        dtype=tf.float32
    )
)


def build_datasets():

    training_dataset = (
        tf.data.Dataset
        .from_generator(
            lambda:
                frame_generator(
                    train_ids,
                    shuffle=True,
                    seed=SEED
                ),
            output_signature=
                OUTPUT_SIGNATURE
        )
        .batch(
            SELECTED_BATCH_SIZE,
            drop_remainder=True
        )
        .prefetch(1)
    )


    validation_dataset = (
        tf.data.Dataset
        .from_generator(
            lambda:
                frame_generator(
                    val_ids,
                    shuffle=False,
                    seed=SEED
                ),
            output_signature=
                OUTPUT_SIGNATURE
        )
        .batch(
            SELECTED_BATCH_SIZE,
            drop_remainder=False
        )
        .prefetch(1)
    )


    train_steps = (
        TRAIN_FRAMES //
        SELECTED_BATCH_SIZE
    )

    validation_steps = int(
        np.ceil(
            VAL_FRAMES /
            SELECTED_BATCH_SIZE
        )
    )


    return (
        training_dataset,
        validation_dataset,
        train_steps,
        validation_steps
    )


_, _, train_steps_check, val_steps_check = (
    build_datasets()
)

print(
    "Training steps/epoch:",
    train_steps_check
)

print(
    "Validation steps:",
    val_steps_check
)

print("\n✓ Fixed data pipeline ready.")

Training steps/epoch: 400
Validation steps: 50

✓ Fixed data pipeline ready.


In [ ]:
SMOOTH = 1e-6


def dice_coefficient(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )


    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
    )


    dice = (
        2.0 * intersection + SMOOTH
    ) / (
        denominator + SMOOTH
    )


    return tf.reduce_mean(
        dice
    )


def iou_metric(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )


    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    union = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
        -
        intersection
    )


    iou = (
        intersection + SMOOTH
    ) / (
        union + SMOOTH
    )


    return tf.reduce_mean(
        iou
    )


def dice_loss(
    y_true,
    y_pred
):

    return (
        1.0 -
        dice_coefficient(
            y_true,
            y_pred
        )
    )


print("✓ Common Dice/IoU components ready.")

✓ Common Dice/IoU components ready.


In [ ]:
# ================================================================
# CONTROL — BCE + DICE
# ================================================================

def bce_dice_loss(
    y_true,
    y_pred
):

    bce = (
        tf.keras.losses
        .binary_crossentropy(
            y_true,
            y_pred
        )
    )

    bce = tf.reduce_mean(
        bce
    )

    return (
        bce +
        dice_loss(
            y_true,
            y_pred
        )
    )


# ================================================================
# FOCAL + DICE
# ================================================================

FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0


def focal_loss_component(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )


    y_pred = tf.clip_by_value(
        y_pred,
        1e-7,
        1.0 - 1e-7
    )


    cross_entropy = (
        -y_true * tf.math.log(y_pred)
        -
        (1.0 - y_true)
        * tf.math.log(
            1.0 - y_pred
        )
    )


    p_t = (
        y_true * y_pred
        +
        (1.0 - y_true)
        * (1.0 - y_pred)
    )


    alpha_factor = (
        y_true * FOCAL_ALPHA
        +
        (1.0 - y_true)
        * (1.0 - FOCAL_ALPHA)
    )


    modulating_factor = (
        tf.pow(
            1.0 - p_t,
            FOCAL_GAMMA
        )
    )


    focal = (
        alpha_factor
        *
        modulating_factor
        *
        cross_entropy
    )


    return tf.reduce_mean(
        focal
    )


def focal_dice_loss(
    y_true,
    y_pred
):

    return (
        focal_loss_component(
            y_true,
            y_pred
        )
        +
        dice_loss(
            y_true,
            y_pred
        )
    )


# ================================================================
# TVERSKY
# ================================================================

TVERSKY_ALPHA = 0.5
TVERSKY_BETA = 0.5


def tversky_index(
    y_true,
    y_pred
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )


    y_true = tf.reshape(
        y_true,
        [-1]
    )

    y_pred = tf.reshape(
        y_pred,
        [-1]
    )


    true_positive = tf.reduce_sum(
        y_true * y_pred
    )

    false_positive = tf.reduce_sum(
        (1.0 - y_true)
        * y_pred
    )

    false_negative = tf.reduce_sum(
        y_true
        * (1.0 - y_pred)
    )


    return (
        true_positive + SMOOTH
    ) / (
        true_positive
        +
        TVERSKY_ALPHA
        * false_positive
        +
        TVERSKY_BETA
        * false_negative
        +
        SMOOTH
    )


def tversky_loss(
    y_true,
    y_pred
):

    return (
        1.0 -
        tversky_index(
            y_true,
            y_pred
        )
    )


LOSS_FUNCTIONS = {
    "bce_dice":
        bce_dice_loss,

    "focal_dice":
        focal_dice_loss,

    "tversky":
        tversky_loss
}


print("✓ Three loss functions defined.")

print("\nLoss arms:")
for name in LOSS_FUNCTIONS:
    print(" -", name)

✓ Three loss functions defined.

Loss arms:
 - bce_dice
 - focal_dice
 - tversky


In [ ]:
def conv_block(
    x,
    filters
):

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    x = tf.keras.layers.Conv2D(
        filters,
        3,
        activation="relu",
        padding="same"
    )(x)

    return x


def build_large_unet(
    input_shape=(256, 256, 1)
):

    filters = SELECTED_FILTERS

    assert filters == [
        32,
        64,
        128,
        256,
        512
    ]

    f1, f2, f3, f4, f5 = filters


    inputs = tf.keras.Input(
        shape=input_shape
    )


    c1 = conv_block(inputs, f1)
    p1 = tf.keras.layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, f2)
    p2 = tf.keras.layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, f3)
    p3 = tf.keras.layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, f4)
    p4 = tf.keras.layers.MaxPooling2D()(c4)

    c5 = conv_block(p4, f5)


    u6 = tf.keras.layers.Conv2DTranspose(
        f4,
        2,
        strides=2,
        padding="same"
    )(c5)

    u6 = tf.keras.layers.Concatenate()(
        [u6, c4]
    )

    c6 = conv_block(u6, f4)


    u7 = tf.keras.layers.Conv2DTranspose(
        f3,
        2,
        strides=2,
        padding="same"
    )(c6)

    u7 = tf.keras.layers.Concatenate()(
        [u7, c3]
    )

    c7 = conv_block(u7, f3)


    u8 = tf.keras.layers.Conv2DTranspose(
        f2,
        2,
        strides=2,
        padding="same"
    )(c7)

    u8 = tf.keras.layers.Concatenate()(
        [u8, c2]
    )

    c8 = conv_block(u8, f2)


    u9 = tf.keras.layers.Conv2DTranspose(
        f1,
        2,
        strides=2,
        padding="same"
    )(c8)

    u9 = tf.keras.layers.Concatenate()(
        [u9, c1]
    )

    c9 = conv_block(u9, f1)


    outputs = tf.keras.layers.Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c9)


    segmentation_model = tf.keras.Model(
        inputs,
        outputs
    )


    return segmentation_model

In [ ]:
tf.keras.backend.clear_session()
gc.collect()

tf.random.set_seed(SEED)

verification_model = (
    build_large_unet(
        input_shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        )
    )
)


actual_parameters = int(
    verification_model.count_params()
)


print(
    "Large U-Net parameters:",
    f"{actual_parameters:,}"
)


assert (
    actual_parameters
    ==
    SELECTED_PARAMETER_COUNT
)

assert (
    actual_parameters
    ==
    7_759_521
)


del verification_model

tf.keras.backend.clear_session()
gc.collect()


print(
    "\n✓ Phase 1C selected architecture "
    "reproduced exactly."
)

Large U-Net parameters: 7,759,521

✓ Phase 1C selected architecture reproduced exactly.


In [ ]:
# ================================================================
# COMMON INITIALIZATION FOR ALL LOSS ARMS
# ================================================================

INITIAL_WEIGHTS_PATH = (
    P1D_MODEL_DIR /
    "loss_experiment_initial_weights.weights.h5"
)


tf.keras.backend.clear_session()
gc.collect()

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


initialization_model = (
    build_large_unet(
        input_shape=(
            SELECTED_RESOLUTION,
            SELECTED_RESOLUTION,
            1
        )
    )
)


assert (
    initialization_model.count_params()
    ==
    7_759_521
)


initialization_model.save_weights(
    INITIAL_WEIGHTS_PATH
)


print("✓ Common initial weights saved:")
print(INITIAL_WEIGHTS_PATH)


del initialization_model

tf.keras.backend.clear_session()
gc.collect()

✓ Common initial weights saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/loss_experiment_initial_weights.weights.h5


0

In [ ]:
loss_protocol = {

    "experiment":
        "Phase 1D — Controlled Loss Function Experiment",

    "status":
        "PROTOCOL FROZEN BEFORE TRAINING",

    "variable_tested":
        "Loss function",

    "loss_arms":
        LOSS_CONFIGS,

    "fixed_settings": {

        "resolution":
            SELECTED_RESOLUTION,

        "architecture":
            "Large U-Net",

        "filters":
            SELECTED_FILTERS,

        "parameters":
            SELECTED_PARAMETER_COUNT,

        "learning_rate":
            SELECTED_LEARNING_RATE,

        "optimizer":
            SELECTED_OPTIMIZER,

        "batch_size":
            SELECTED_BATCH_SIZE,

        "augmentation":
            False,

        "epochs":
            MAX_EPOCHS,

        "threshold":
            THRESHOLD,

        "seed":
            SEED
    },

    "initialization_control":
        (
            "All three loss arms start from "
            "the exact same saved initial weights."
        ),

    "data_order_control":
        (
            "All three arms use the same seeded "
            "training frame-order generator."
        ),

    "train_patients":
        len(train_ids),

    "validation_patients":
        len(val_ids),

    "train_frames_per_epoch":
        TRAIN_FRAMES,

    "validation_frames":
        VAL_FRAMES,

    "selection_dataset":
        "Validation set only",

    "test_used":
        False,

    "selection_metrics": [
        "overall Dice",
        "ED Dice",
        "ES Dice",
        "IoU",
        "normalized HD95",
        "ED area MAE",
        "ES area MAE",
        "overall area MAE"
    ]
}


LOSS_PROTOCOL_PATH = (
    P1D_CONFIG_DIR /
    "phase1D_loss_protocol.json"
)


with open(
    LOSS_PROTOCOL_PATH,
    "w"
) as f:

    json.dump(
        loss_protocol,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 1D PROTOCOL FROZEN")
print("=" * 70)

print("\nLosses:")
print("1. BCE + Dice")
print("2. Focal + Dice")
print("3. Tversky")

print("\nArchitecture:")
print(
    f"Large U-Net — "
    f"{SELECTED_PARAMETER_COUNT:,} parameters"
)

print("\nCommon initialization: YES")
print("Same training order    : YES")
print("Validation only        : YES")
print("Test used              : NO")

print("\nSaved:")
print(LOSS_PROTOCOL_PATH)

print("\n✓ Ready for controlled loss training.")

PHASE 1D PROTOCOL FROZEN

Losses:
1. BCE + Dice
2. Focal + Dice
3. Tversky

Architecture:
Large U-Net — 7,759,521 parameters

Common initialization: YES
Same training order    : YES
Validation only        : YES
Test used              : NO

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/configs/phase1D_loss_protocol.json

✓ Ready for controlled loss training.


In [ ]:
# ================================================================
# PHASE 1D — CELL 17
# CONTROLLED LOSS TRAINING FUNCTION
# ================================================================

def train_loss_experiment(loss_name):

    assert loss_name in LOSS_FUNCTIONS

    print("\n" + "=" * 70)
    print(
        f"TRAINING LOSS: "
        f"{loss_name.upper()}"
    )
    print("=" * 70)

    # ------------------------------------------------------------
    # CLEAN MEMORY
    # ------------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    # ------------------------------------------------------------
    # RESET RANDOM STATE
    # ------------------------------------------------------------

    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    # ------------------------------------------------------------
    # DATASETS
    # ------------------------------------------------------------

    (
        training_dataset,
        validation_dataset,
        train_steps,
        validation_steps
    ) = build_datasets()

    print(
        "Training steps/epoch:",
        train_steps
    )

    print(
        "Validation steps:",
        validation_steps
    )

    # ------------------------------------------------------------
    # BUILD IDENTICAL LARGE U-NET
    # ------------------------------------------------------------

    segmentation_model = (
        build_large_unet(
            input_shape=(
                SELECTED_RESOLUTION,
                SELECTED_RESOLUTION,
                1
            )
        )
    )

    assert (
        segmentation_model.count_params()
        ==
        SELECTED_PARAMETER_COUNT
    )

    # ------------------------------------------------------------
    # LOAD EXACT SAME INITIAL WEIGHTS
    # ------------------------------------------------------------

    segmentation_model.load_weights(
        INITIAL_WEIGHTS_PATH
    )

    print(
        "✓ Common initial weights loaded."
    )

    # ------------------------------------------------------------
    # OPTIMIZER
    # ------------------------------------------------------------

    optimizer_instance = (
        tf.keras.optimizers.Adam(
            learning_rate=
                SELECTED_LEARNING_RATE
        )
    )

    # ------------------------------------------------------------
    # SELECT LOSS
    # ------------------------------------------------------------

    selected_loss_function = (
        LOSS_FUNCTIONS[
            loss_name
        ]
    )

    segmentation_model.compile(
        optimizer=optimizer_instance,
        loss=selected_loss_function,
        metrics=[
            dice_coefficient,
            iou_metric
        ]
    )

    # ------------------------------------------------------------
    # OUTPUT PATHS
    # ------------------------------------------------------------

    model_path = (
        P1D_MODEL_DIR /
        f"{loss_name}_best.keras"
    )

    history_path = (
        P1D_HISTORY_DIR /
        f"{loss_name}_history.csv"
    )

    # ------------------------------------------------------------
    # CHECKPOINT
    # ------------------------------------------------------------

    checkpoint = (
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(model_path),
            monitor=
                "val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    )

    # ------------------------------------------------------------
    # TRAIN
    # ------------------------------------------------------------

    start_time = time.time()

    history = segmentation_model.fit(
        training_dataset,
        validation_data=
            validation_dataset,
        epochs=MAX_EPOCHS,
        steps_per_epoch=
            train_steps,
        validation_steps=
            validation_steps,
        callbacks=[
            checkpoint
        ],
        verbose=1
    )

    training_minutes = (
        time.time() -
        start_time
    ) / 60.0

    # ------------------------------------------------------------
    # SAVE HISTORY
    # ------------------------------------------------------------

    history_df = pd.DataFrame(
        history.history
    )

    history_df.index = (
        np.arange(
            len(history_df)
        ) + 1
    )

    history_df.index.name = "epoch"

    history_df.to_csv(
        history_path
    )

    # ------------------------------------------------------------
    # BEST EPOCH
    # ------------------------------------------------------------

    val_dice_values = np.asarray(
        history.history[
            "val_dice_coefficient"
        ],
        dtype=float
    )

    best_index = int(
        np.argmax(
            val_dice_values
        )
    )

    result = {

        "loss_name":
            loss_name,

        "parameters":
            int(
                segmentation_model.count_params()
            ),

        "best_epoch":
            int(best_index + 1),

        "best_val_dice_training":
            float(
                val_dice_values[
                    best_index
                ]
            ),

        "val_iou_at_best_dice_epoch":
            float(
                history.history[
                    "val_iou_metric"
                ][best_index]
            ),

        "training_minutes":
            float(training_minutes),

        "model_path":
            str(model_path),

        "history_path":
            str(history_path)
    }

    # ------------------------------------------------------------
    # REPORT
    # ------------------------------------------------------------

    print("\n" + "-" * 70)

    print(
        f"{loss_name.upper()} COMPLETE"
    )

    print(
        "Parameters:",
        f"{result['parameters']:,}"
    )

    print(
        "Best epoch:",
        result["best_epoch"]
    )

    print(
        "Best soft validation Dice:",
        f"{result['best_val_dice_training']:.6f}"
    )

    print(
        "Validation IoU at best Dice:",
        f"{result['val_iou_at_best_dice_epoch']:.6f}"
    )

    print(
        "Training time:",
        f"{result['training_minutes']:.2f} minutes"
    )

    print(
        "Checkpoint:",
        model_path
    )

    print("-" * 70)

    # ------------------------------------------------------------
    # CLEAN MEMORY
    # ------------------------------------------------------------

    del segmentation_model
    del optimizer_instance
    del history
    del training_dataset
    del validation_dataset

    tf.keras.backend.clear_session()
    gc.collect()

    return result

In [ ]:
bce_dice_result = (
    train_loss_experiment(
        "bce_dice"
    )
)

bce_dice_result


TRAINING LOSS: BCE_DICE
Training steps/epoch: 400
Validation steps: 50
✓ Common initial weights loaded.
Epoch 1/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - dice_coefficient: 0.3204 - iou_metric: 0.2122 - loss: 1.0459
Epoch 1: val_dice_coefficient improved from None to 0.82047, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/bce_dice_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/bce_dice_best.keras
400/400 ━━━━━━━━━━━━━━━━━━━━ 42s 84ms/step - dice_coefficient: 0.5133 - iou_metric: 0.3781 - loss: 0.7094 - val_dice_coefficient: 0.8205 - val_iou_metric: 0.7009 - val_loss: 0.2677
Epoch 2/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - dice_coefficient: 0.8141 - iou_metric: 0.6952 - loss: 0.2720
Epoch 2: val_dice_coefficient improved from 0.82047 to 0.85203, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/bce_dice_best.keras

Epoc

{'loss_name': 'bce_dice',
 'parameters': 7759521,
 'best_epoch': 28,
 'best_val_dice_training': 0.9335587024688721,
 'val_iou_at_best_dice_epoch': 0.8774716258049011,
 'training_minutes': 16.992635901769003,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/bce_dice_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/histories/bce_dice_history.csv'}

In [ ]:
focal_dice_result = (
    train_loss_experiment(
        "focal_dice"
    )
)

focal_dice_result


TRAINING LOSS: FOCAL_DICE
Training steps/epoch: 400
Validation steps: 50
✓ Common initial weights loaded.
Epoch 1/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - dice_coefficient: 0.3584 - iou_metric: 0.2504 - loss: 0.7105
Epoch 1: val_dice_coefficient improved from None to 0.81263, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/focal_dice_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/focal_dice_best.keras
400/400 ━━━━━━━━━━━━━━━━━━━━ 46s 88ms/step - dice_coefficient: 0.5707 - iou_metric: 0.4422 - loss: 0.4835 - val_dice_coefficient: 0.8126 - val_iou_metric: 0.6903 - val_loss: 0.2168
Epoch 2/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - dice_coefficient: 0.8206 - iou_metric: 0.7079 - loss: 0.2129
Epoch 2: val_dice_coefficient improved from 0.81263 to 0.85123, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/focal_dice_best.ker

{'loss_name': 'focal_dice',
 'parameters': 7759521,
 'best_epoch': 23,
 'best_val_dice_training': 0.9332960247993469,
 'val_iou_at_best_dice_epoch': 0.8771555423736572,
 'training_minutes': 17.02406718333562,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/focal_dice_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/histories/focal_dice_history.csv'}

In [ ]:
tversky_result = (
    train_loss_experiment(
        "tversky"
    )
)

tversky_result


TRAINING LOSS: TVERSKY
Training steps/epoch: 400
Validation steps: 50
✓ Common initial weights loaded.
Epoch 1/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - dice_coefficient: 0.3624 - iou_metric: 0.2514 - loss: 0.6332
Epoch 1: val_dice_coefficient improved from None to 0.80437, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/tversky_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/tversky_best.keras
400/400 ━━━━━━━━━━━━━━━━━━━━ 39s 81ms/step - dice_coefficient: 0.5602 - iou_metric: 0.4284 - loss: 0.4343 - val_dice_coefficient: 0.8044 - val_iou_metric: 0.6807 - val_loss: 0.1913
Epoch 2/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - dice_coefficient: 0.7948 - iou_metric: 0.6707 - loss: 0.1993
Epoch 2: val_dice_coefficient improved from 0.80437 to 0.83444, saving model to /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/tversky_best.keras

Epoch 2:

{'loss_name': 'tversky',
 'parameters': 7759521,
 'best_epoch': 30,
 'best_val_dice_training': 0.9318894147872925,
 'val_iou_at_best_dice_epoch': 0.8755056262016296,
 'training_minutes': 16.70666071573893,
 'model_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/tversky_best.keras',
 'history_path': '/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/histories/tversky_history.csv'}

In [ ]:
# ================================================================
# PHASE 1D — CELL 21
# LOSS TRAINING SUMMARY
# ================================================================

loss_training_summary = pd.DataFrame([

    bce_dice_result,
    focal_dice_result,
    tversky_result

])


LOSS_TRAINING_SUMMARY_PATH = (
    P1D_TABLE_DIR /
    "loss_training_summary.csv"
)


loss_training_summary.to_csv(
    LOSS_TRAINING_SUMMARY_PATH,
    index=False
)


display_columns = [

    "loss_name",
    "parameters",
    "best_epoch",
    "best_val_dice_training",
    "val_iou_at_best_dice_epoch",
    "training_minutes"

]


display(
    loss_training_summary[
        display_columns
    ].round(6)
)


print(
    "\n✓ Loss training summary saved:"
)

print(
    LOSS_TRAINING_SUMMARY_PATH
)

,loss_name,parameters,best_epoch,best_val_dice_training,val_iou_at_best_dice_epoch,training_minutes
0,bce_dice,7759521,28,0.933559,0.877472,16.992636
1,focal_dice,7759521,23,0.933296,0.877156,17.024067
2,tversky,7759521,30,0.931889,0.875506,16.706661



✓ Loss training summary saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/tables/loss_training_summary.csv


In [ ]:
# ================================================================
# PHASE 1D — CELL 22
# TRAINING INTEGRITY CHECKS
# ================================================================

expected_losses = [
    "bce_dice",
    "focal_dice",
    "tversky"
]


for loss_name in expected_losses:

    row = (
        loss_training_summary[
            loss_training_summary[
                "loss_name"
            ] == loss_name
        ].iloc[0]
    )

    assert (
        int(row["parameters"])
        ==
        7_759_521
    )

    checkpoint_path = (
        P1D_MODEL_DIR /
        f"{loss_name}_best.keras"
    )

    assert checkpoint_path.exists()

    history_path = (
        P1D_HISTORY_DIR /
        f"{loss_name}_history.csv"
    )

    assert history_path.exists()


print(
    "✓ All models contain exactly "
    "7,759,521 parameters."
)

print(
    "✓ All three best checkpoints exist."
)

print(
    "✓ All three training histories exist."
)

print(
    "✓ Common initialization was used."
)

print(
    "✓ Same seeded data-order protocol was used."
)

print(
    "✓ Test set was NOT used."
)

print(
    "\nPhase 1D training stage complete."
)

✓ All models contain exactly 7,759,521 parameters.
✓ All three best checkpoints exist.
✓ All three training histories exist.
✓ Common initialization was used.
✓ Same seeded data-order protocol was used.
✓ Test set was NOT used.

Phase 1D training stage complete.


In [ ]:
# ================================================================
# PHASE 1D — CELL 23
# POSTPROCESSING + VALIDATION METRICS
# ================================================================

def keep_largest_component(mask):

    mask = np.asarray(mask).astype(np.uint8)

    if mask.sum() == 0:
        return np.zeros_like(mask, dtype=bool)

    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    if n_labels <= 1:
        return np.zeros_like(mask, dtype=bool)

    largest_label = (
        1 +
        np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )
    )

    return labels == largest_label


def postprocess_prediction(mask):

    largest = keep_largest_component(mask)

    filled = binary_fill_holes(
        largest
    )

    return np.asarray(
        filled,
        dtype=bool
    )


def binary_dice(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    denominator = gt.sum() + pred.sum()

    if denominator == 0:
        return 1.0

    return (
        2.0 * intersection /
        denominator
    )


def binary_iou(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    intersection = np.logical_and(
        gt, pred
    ).sum()

    union = np.logical_or(
        gt, pred
    ).sum()

    if union == 0:
        return 1.0

    return intersection / union


def binary_precision(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(
        gt, pred
    ).sum()

    fp = np.logical_and(
        ~gt, pred
    ).sum()

    if (tp + fp) == 0:
        return (
            1.0
            if gt.sum() == 0
            else 0.0
        )

    return tp / (tp + fp)


def binary_recall(gt, pred):

    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)

    tp = np.logical_and(
        gt, pred
    ).sum()

    fn = np.logical_and(
        gt, ~pred
    ).sum()

    if (tp + fn) == 0:
        return 1.0

    return tp / (tp + fn)


def boundary_points(mask):

    mask = np.asarray(mask).astype(np.uint8)

    kernel = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    eroded = cv2.erode(
        mask,
        kernel,
        iterations=1
    )

    boundary = (
        mask.astype(bool)
        &
        ~eroded.astype(bool)
    )

    return np.column_stack(
        np.where(boundary)
    )


def hd95_pixels(gt, pred):

    gt_points = boundary_points(gt)
    pred_points = boundary_points(pred)

    if (
        len(gt_points) == 0
        or len(pred_points) == 0
    ):
        return np.nan

    distances = cdist(
        gt_points,
        pred_points
    )

    gt_to_pred = distances.min(
        axis=1
    )

    pred_to_gt = distances.min(
        axis=0
    )

    all_distances = np.concatenate([
        gt_to_pred,
        pred_to_gt
    ])

    return float(
        np.percentile(
            all_distances,
            95
        )
    )


print(
    "✓ Postprocessing and validation metrics ready."
)

✓ Postprocessing and validation metrics ready.


In [ ]:
# ================================================================
# PHASE 1D — CELL 24
# LOSS CHECKPOINT EVALUATOR
# ================================================================

def evaluate_loss_checkpoint(
    loss_name
):

    assert loss_name in [
        "bce_dice",
        "focal_dice",
        "tversky"
    ]

    model_path = (
        P1D_MODEL_DIR /
        f"{loss_name}_best.keras"
    )

    assert model_path.exists(), (
        f"Checkpoint missing:\n{model_path}"
    )


    print("\n" + "=" * 70)
    print(
        f"EVALUATING LOSS: "
        f"{loss_name.upper()}"
    )
    print("=" * 70)


    tf.keras.backend.clear_session()
    gc.collect()


    evaluation_model = (
        tf.keras.models.load_model(
            model_path,
            custom_objects={
                "bce_dice_loss":
                    bce_dice_loss,

                "focal_dice_loss":
                    focal_dice_loss,

                "focal_loss_component":
                    focal_loss_component,

                "tversky_loss":
                    tversky_loss,

                "tversky_index":
                    tversky_index,

                "dice_coefficient":
                    dice_coefficient,

                "iou_metric":
                    iou_metric,

                "dice_loss":
                    dice_loss
            }
        )
    )


    assert (
        evaluation_model.count_params()
        ==
        7_759_521
    )


    records = []


    for patient_number, pid in enumerate(
        val_ids,
        start=1
    ):

        for phase in ["ED", "ES"]:

            image, gt_mask = (
                load_camus_frame(
                    pid,
                    phase
                )
            )


            x, gt_resized = (
                preprocess_frame(
                    image,
                    gt_mask
                )
            )


            probability = (
                evaluation_model(
                    x[None, ...],
                    training=False
                )
                .numpy()[0, :, :, 0]
            )


            pred_raw = (
                probability >= THRESHOLD
            )


            pred_mask = (
                postprocess_prediction(
                    pred_raw
                )
            )


            gt_binary = (
                gt_resized[:, :, 0]
                > 0.5
            )


            gt_area = int(
                gt_binary.sum()
            )

            pred_area = int(
                pred_mask.sum()
            )


            if gt_area > 0:

                signed_area_error_pct = (
                    (pred_area - gt_area)
                    / gt_area
                    * 100.0
                )

                abs_area_error_pct = abs(
                    signed_area_error_pct
                )

            else:

                signed_area_error_pct = np.nan
                abs_area_error_pct = np.nan


            hd95 = hd95_pixels(
                gt_binary,
                pred_mask
            )


            records.append({

                "patient_id":
                    pid,

                "phase":
                    phase,

                "loss_name":
                    loss_name,

                "dice":
                    binary_dice(
                        gt_binary,
                        pred_mask
                    ),

                "iou":
                    binary_iou(
                        gt_binary,
                        pred_mask
                    ),

                "precision":
                    binary_precision(
                        gt_binary,
                        pred_mask
                    ),

                "recall":
                    binary_recall(
                        gt_binary,
                        pred_mask
                    ),

                "hd95_pixels":
                    hd95,

                "normalized_hd95":
                    (
                        hd95 /
                        SELECTED_RESOLUTION
                        if np.isfinite(hd95)
                        else np.nan
                    ),

                "gt_area_pixels":
                    gt_area,

                "pred_area_pixels":
                    pred_area,

                "signed_area_error_pct":
                    signed_area_error_pct,

                "abs_area_error_pct":
                    abs_area_error_pct
            })


        if patient_number % 10 == 0:

            print(
                f"Evaluated "
                f"{patient_number}/"
                f"{len(val_ids)} patients"
            )


    results_df = pd.DataFrame(
        records
    )

    assert len(results_df) == 100


    output_path = (
        P1D_TABLE_DIR /
        f"{loss_name}_validation_metrics.csv"
    )


    results_df.to_csv(
        output_path,
        index=False
    )


    print(
        "\nFrames evaluated:",
        len(results_df)
    )

    print("Saved:")
    print(output_path)


    del evaluation_model

    tf.keras.backend.clear_session()
    gc.collect()


    return results_df

In [ ]:
# ================================================================
# PHASE 1D — CELL 25
# EVALUATE ALL LOSS CHECKPOINTS
# ================================================================

bce_dice_eval = (
    evaluate_loss_checkpoint(
        "bce_dice"
    )
)


focal_dice_eval = (
    evaluate_loss_checkpoint(
        "focal_dice"
    )
)


tversky_eval = (
    evaluate_loss_checkpoint(
        "tversky"
    )
)


assert len(bce_dice_eval) == 100
assert len(focal_dice_eval) == 100
assert len(tversky_eval) == 100


print(
    "\n✓ All three loss checkpoints evaluated."
)

print(
    "✓ Same 100 validation frames used."
)

print(
    "✓ Test set was NOT used."
)


EVALUATING LOSS: BCE_DICE
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/tables/bce_dice_validation_metrics.csv

EVALUATING LOSS: FOCAL_DICE
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/tables/focal_dice_validation_metrics.csv

EVALUATING LOSS: TVERSKY
Evaluated 10/50 patients
Evaluated 20/50 patients
Evaluated 30/50 patients
Evaluated 40/50 patients
Evaluated 50/50 patients

Frames evaluated: 100
Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/tables/tversky_validation_metrics.csv

✓ All three loss checkpoints evaluated.
✓ Same 100 validation frames used.
✓ Test set was NOT used.


In [ ]:
# ================================================================
# PHASE 1D — CELL 26
# FINAL LOSS VALIDATION COMPARISON
# ================================================================

def summarize_loss_results(
    evaluation_df,
    training_result
):

    ed = evaluation_df[
        evaluation_df["phase"] == "ED"
    ]

    es = evaluation_df[
        evaluation_df["phase"] == "ES"
    ]


    return {

        "loss_name":
            training_result[
                "loss_name"
            ],

        "parameters":
            training_result[
                "parameters"
            ],

        "best_epoch":
            training_result[
                "best_epoch"
            ],

        "training_minutes":
            training_result[
                "training_minutes"
            ],

        "overall_dice":
            evaluation_df[
                "dice"
            ].mean(),

        "ED_dice":
            ed[
                "dice"
            ].mean(),

        "ES_dice":
            es[
                "dice"
            ].mean(),

        "overall_iou":
            evaluation_df[
                "iou"
            ].mean(),

        "precision":
            evaluation_df[
                "precision"
            ].mean(),

        "recall":
            evaluation_df[
                "recall"
            ].mean(),

        "normalized_hd95":
            evaluation_df[
                "normalized_hd95"
            ].mean(),

        "ED_area_MAE_pct":
            ed[
                "abs_area_error_pct"
            ].mean(),

        "ES_area_MAE_pct":
            es[
                "abs_area_error_pct"
            ].mean(),

        "overall_area_MAE_pct":
            evaluation_df[
                "abs_area_error_pct"
            ].mean(),

        "ED_signed_area_error_pct":
            ed[
                "signed_area_error_pct"
            ].mean(),

        "ES_signed_area_error_pct":
            es[
                "signed_area_error_pct"
            ].mean(),

        "overall_signed_area_error_pct":
            evaluation_df[
                "signed_area_error_pct"
            ].mean()
    }


loss_comparison = pd.DataFrame([

    summarize_loss_results(
        bce_dice_eval,
        bce_dice_result
    ),

    summarize_loss_results(
        focal_dice_eval,
        focal_dice_result
    ),

    summarize_loss_results(
        tversky_eval,
        tversky_result
    )

])


LOSS_COMPARISON_PATH = (
    P1D_TABLE_DIR /
    "loss_validation_comparison.csv"
)


loss_comparison.to_csv(
    LOSS_COMPARISON_PATH,
    index=False
)


display(
    loss_comparison.round(4)
)


print(
    "\n✓ Final loss comparison saved:"
)

print(
    LOSS_COMPARISON_PATH
)

,loss_name,parameters,best_epoch,training_minutes,overall_dice,ED_dice,ES_dice,overall_iou,precision,recall,normalized_hd95,ED_area_MAE_pct,ES_area_MAE_pct,overall_area_MAE_pct,ED_signed_area_error_pct,ES_signed_area_error_pct,overall_signed_area_error_pct
0,bce_dice,7759521,28,16.9926,0.9350,0.9443,0.9257,0.8801,0.9395,0.9356,0.0217,5.8389,9.5696,7.7042,-0.9072,1.6548,0.3738
1,focal_dice,7759521,23,17.0241,0.9351,0.9436,0.9266,0.8804,0.9263,0.9495,0.0221,6.8735,10.0092,8.4414,2.0990,4.9056,3.5023
2,tversky,7759521,30,16.7067,0.9320,0.9402,0.9239,0.8757,0.9249,0.9457,0.0232,6.5612,10.9030,8.7321,0.5975,6.4053,3.5014



✓ Final loss comparison saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/tables/loss_validation_comparison.csv


In [ ]:
# ================================================================
# PHASE 1D — CELL 27
# PAIRED COMPARISON AGAINST BCE + DICE
# ================================================================

def paired_loss_comparison(
    candidate_df,
    candidate_name
):

    assert (
        bce_dice_eval[
            ["patient_id", "phase"]
        ]
        .reset_index(drop=True)
        .equals(
            candidate_df[
                ["patient_id", "phase"]
            ]
            .reset_index(drop=True)
        )
    )


    paired = pd.DataFrame({

        "patient_id":
            bce_dice_eval[
                "patient_id"
            ].values,

        "phase":
            bce_dice_eval[
                "phase"
            ].values,

        "dice_change":
            (
                candidate_df[
                    "dice"
                ].values
                -
                bce_dice_eval[
                    "dice"
                ].values
            ),

        "iou_change":
            (
                candidate_df[
                    "iou"
                ].values
                -
                bce_dice_eval[
                    "iou"
                ].values
            ),

        "hd95_change":
            (
                candidate_df[
                    "normalized_hd95"
                ].values
                -
                bce_dice_eval[
                    "normalized_hd95"
                ].values
            ),

        "area_MAE_change":
            (
                candidate_df[
                    "abs_area_error_pct"
                ].values
                -
                bce_dice_eval[
                    "abs_area_error_pct"
                ].values
            )
    })


    output_path = (
        P1D_TABLE_DIR /
        f"{candidate_name}_vs_bce_dice_paired.csv"
    )

    paired.to_csv(
        output_path,
        index=False
    )


    return {

        "candidate":
            candidate_name,

        "mean_dice_change":
            paired[
                "dice_change"
            ].mean(),

        "mean_iou_change":
            paired[
                "iou_change"
            ].mean(),

        "mean_hd95_change":
            paired[
                "hd95_change"
            ].mean(),

        "mean_area_MAE_change":
            paired[
                "area_MAE_change"
            ].mean(),

        "frames_dice_improved":
            int(
                (
                    paired[
                        "dice_change"
                    ] > 0
                ).sum()
            ),

        "frames_dice_worsened":
            int(
                (
                    paired[
                        "dice_change"
                    ] < 0
                ).sum()
            ),

        "frames_area_error_reduced":
            int(
                (
                    paired[
                        "area_MAE_change"
                    ] < 0
                ).sum()
            )
    }


focal_vs_control = (
    paired_loss_comparison(
        focal_dice_eval,
        "focal_dice"
    )
)


tversky_vs_control = (
    paired_loss_comparison(
        tversky_eval,
        "tversky"
    )
)


paired_loss_summary = pd.DataFrame([
    focal_vs_control,
    tversky_vs_control
])


display(
    paired_loss_summary.round(6)
)


print(
    "\n✓ Paired loss comparisons saved."
)

,candidate,mean_dice_change,mean_iou_change,mean_hd95_change,mean_area_MAE_change,frames_dice_improved,frames_dice_worsened,frames_area_error_reduced
0,focal_dice,0.000078,0.000314,0.000468,0.737146,53,47,50
1,tversky,-0.002971,-0.004385,0.001510,1.027865,46,54,42



✓ Paired loss comparisons saved.


In [ ]:
# ================================================================
# PHASE 1D — CELL 28
# FINAL LOSS FUNCTION DECISION
# ================================================================

SELECTED_LOSS = "bce_dice"

selected_row = (
    loss_comparison[
        loss_comparison["loss_name"]
        == SELECTED_LOSS
    ].iloc[0]
)

focal_row = (
    loss_comparison[
        loss_comparison["loss_name"]
        == "focal_dice"
    ].iloc[0]
)

tversky_row = (
    loss_comparison[
        loss_comparison["loss_name"]
        == "tversky"
    ].iloc[0]
)


loss_decision = {

    "experiment":
        "Phase 1D — Controlled Loss Function Experiment",

    "status":
        "COMPLETED",

    "selected_loss":
        "BCE + Dice",

    "selected_loss_key":
        SELECTED_LOSS,

    "losses_compared": {
        "bce_dice":
            "Binary cross-entropy + Dice loss",

        "focal_dice":
            "Focal loss + Dice loss",

        "tversky":
            "Standalone Tversky loss"
    },

    "fixed_loss_parameters": {
        "focal_alpha": 0.25,
        "focal_gamma": 2.0,
        "tversky_alpha": 0.5,
        "tversky_beta": 0.5
    },

    "fixed_settings": {

        "resolution":
            SELECTED_RESOLUTION,

        "architecture":
            "Large U-Net",

        "filters":
            SELECTED_FILTERS,

        "parameters":
            SELECTED_PARAMETER_COUNT,

        "learning_rate":
            SELECTED_LEARNING_RATE,

        "optimizer":
            SELECTED_OPTIMIZER,

        "batch_size":
            SELECTED_BATCH_SIZE,

        "augmentation":
            False,

        "epochs":
            MAX_EPOCHS,

        "threshold":
            THRESHOLD,

        "seed":
            SEED
    },

    "selected_validation_results": {

        "overall_dice":
            float(
                selected_row["overall_dice"]
            ),

        "ED_dice":
            float(
                selected_row["ED_dice"]
            ),

        "ES_dice":
            float(
                selected_row["ES_dice"]
            ),

        "overall_iou":
            float(
                selected_row["overall_iou"]
            ),

        "precision":
            float(
                selected_row["precision"]
            ),

        "recall":
            float(
                selected_row["recall"]
            ),

        "normalized_hd95":
            float(
                selected_row["normalized_hd95"]
            ),

        "ED_area_MAE_pct":
            float(
                selected_row["ED_area_MAE_pct"]
            ),

        "ES_area_MAE_pct":
            float(
                selected_row["ES_area_MAE_pct"]
            ),

        "overall_area_MAE_pct":
            float(
                selected_row[
                    "overall_area_MAE_pct"
                ]
            ),

        "ED_signed_area_error_pct":
            float(
                selected_row[
                    "ED_signed_area_error_pct"
                ]
            ),

        "ES_signed_area_error_pct":
            float(
                selected_row[
                    "ES_signed_area_error_pct"
                ]
            ),

        "overall_signed_area_error_pct":
            float(
                selected_row[
                    "overall_signed_area_error_pct"
                ]
            )
    },

    "focal_vs_bce_dice_paired": {
        "mean_dice_change":
            float(
                focal_vs_control[
                    "mean_dice_change"
                ]
            ),

        "mean_iou_change":
            float(
                focal_vs_control[
                    "mean_iou_change"
                ]
            ),

        "mean_hd95_change":
            float(
                focal_vs_control[
                    "mean_hd95_change"
                ]
            ),

        "mean_area_MAE_change_pct":
            float(
                focal_vs_control[
                    "mean_area_MAE_change"
                ]
            ),

        "frames_dice_improved":
            int(
                focal_vs_control[
                    "frames_dice_improved"
                ]
            ),

        "frames_dice_worsened":
            int(
                focal_vs_control[
                    "frames_dice_worsened"
                ]
            ),

        "frames_area_error_reduced":
            int(
                focal_vs_control[
                    "frames_area_error_reduced"
                ]
            )
    },

    "selection_dataset":
        "Validation set only",

    "test_set_used":
        False,

    "selection_reason": (
        "BCE + Dice was retained because it provided the strongest "
        "balance between segmentation overlap, boundary accuracy, and "
        "LV area fidelity. Focal + Dice produced only a negligible "
        "increase in mean Dice while increasing normalized HD95 and "
        "area error. BCE + Dice achieved lower ED, ES, and overall "
        "area MAE and substantially lower overall signed area bias. "
        "These properties are particularly relevant because subsequent "
        "quantitative LV analysis depends on accurate mask geometry "
        "and area estimation."
    ),

    "interpretation_note": (
        "The loss comparison was controlled using the same Large U-Net "
        "architecture, identical initial weights, the same seeded data "
        "order, and the same training budget. The result applies to the "
        "specific predefined loss configurations tested here; it does "
        "not imply that all possible focal or Tversky configurations "
        "would perform worse."
    ),

    "selected_checkpoint":
        str(
            P1D_MODEL_DIR /
            "bce_dice_best.keras"
        ),

    "next_stage":
        "Phase 1E — Final Segmentation Validation"
}


LOSS_DECISION_PATH = (
    P1D_CONFIG_DIR /
    "phase1D_loss_decision.json"
)


with open(
    LOSS_DECISION_PATH,
    "w"
) as f:

    json.dump(
        loss_decision,
        f,
        indent=2
    )


print("=" * 70)
print("PHASE 1D COMPLETED")
print("=" * 70)

print("\nSELECTED LOSS:")
print("BCE + Dice")

print("\nSELECTED VALIDATION PERFORMANCE")
print(
    f"Overall Dice      : "
    f"{selected_row['overall_dice']:.4f}"
)
print(
    f"ED Dice           : "
    f"{selected_row['ED_dice']:.4f}"
)
print(
    f"ES Dice           : "
    f"{selected_row['ES_dice']:.4f}"
)
print(
    f"Overall IoU       : "
    f"{selected_row['overall_iou']:.4f}"
)
print(
    f"Normalized HD95   : "
    f"{selected_row['normalized_hd95']:.4f}"
)
print(
    f"ED Area MAE       : "
    f"{selected_row['ED_area_MAE_pct']:.2f}%"
)
print(
    f"ES Area MAE       : "
    f"{selected_row['ES_area_MAE_pct']:.2f}%"
)
print(
    f"Overall Area MAE  : "
    f"{selected_row['overall_area_MAE_pct']:.2f}%"
)
print(
    f"Overall Area Bias : "
    f"{selected_row['overall_signed_area_error_pct']:+.2f}%"
)

print("\nTEST SET USED: NO")

print("\n" + "=" * 70)
print("FINAL SELECTED SEGMENTATION CONFIGURATION")
print("=" * 70)

print("Resolution    : 256×256")
print("Architecture  : Large U-Net")
print(
    f"Parameters    : "
    f"{SELECTED_PARAMETER_COUNT:,}"
)
print("Learning rate : 1e-4")
print("Optimizer     : Adam")
print("Batch size    : 2")
print("Augmentation  : None")
print("Loss          : BCE + Dice")
print("Threshold     : 0.50")
print("Seed          : 42")

print("\nDecision saved:")
print(LOSS_DECISION_PATH)

print("\nSelected checkpoint:")
print(
    P1D_MODEL_DIR /
    "bce_dice_best.keras"
)

print("\nNEXT:")
print(
    "Phase 1E — Final Segmentation Validation"
)

print("\n✓ Phase 1D is now complete.")

PHASE 1D COMPLETED

SELECTED LOSS:
BCE + Dice

SELECTED VALIDATION PERFORMANCE
Overall Dice      : 0.9350
ED Dice           : 0.9443
ES Dice           : 0.9257
Overall IoU       : 0.8801
Normalized HD95   : 0.0217
ED Area MAE       : 5.84%
ES Area MAE       : 9.57%
Overall Area MAE  : 7.70%
Overall Area Bias : +0.37%

TEST SET USED: NO

FINAL SELECTED SEGMENTATION CONFIGURATION
Resolution    : 256×256
Architecture  : Large U-Net
Parameters    : 7,759,521
Learning rate : 1e-4
Optimizer     : Adam
Batch size    : 2
Augmentation  : None
Loss          : BCE + Dice
Threshold     : 0.50
Seed          : 42

Decision saved:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/configs/phase1D_loss_decision.json

Selected checkpoint:
/content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation/phase1D_loss/models/bce_dice_best.keras

NEXT:
Phase 1E — Final Segmentation Validation

✓ Phase 1D is now complete.
